# ReelIntel — Fish ID model training (Colab)

This notebook fine-tunes a MobileNetV3-Small classifier on the ZIP
exported from the admin **Training → Export** tab and produces
three artifacts you upload back at **Training → Models**:

- `fish_id_model.tflite` — quantized on-device classifier
- `fish_id_labels.json` — label index + thresholds + excluded species
- `fish_id_metrics.json` — per-species accuracy, confusion matrix, lookalike-group breakdown

**Set the runtime to GPU**: Runtime → Change runtime type → T4 GPU.
Then run all cells top-to-bottom. Expect 5–10 minutes end-to-end for
a few thousand images.


In [ ]:
# 1. Upload the export ZIP.
# When this cell runs, Colab will prompt you to pick a file.
from google.colab import files
print('Pick the reelintel-training-*.zip you downloaded from admin.')
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one ZIP.'
EXPORT_ZIP = list(uploaded.keys())[0]
print(f'Got: {EXPORT_ZIP} ({len(next(iter(uploaded.values())))/1024/1024:.1f} MB)')

In [ ]:
# 2. Fetch the training script from the repo and run it.
# (Skip this cell if you'd rather copy-paste train_fish_id.py inline.)
!git clone --quiet https://github.com/robertboot/know-your-catch /content/kyc || true
!cp /content/kyc/training/train_fish_id.py /content/train_fish_id.py
!pip install --quiet 'tensorflow==2.15.*' pillow numpy scikit-learn
!mkdir -p /content/artifacts

In [ ]:
# 3. Train.
!python /content/train_fish_id.py \
  --export /content/$EXPORT_ZIP \
  --out    /content/artifacts \
  --epochs 20

In [ ]:
# 4. Pack the three artifacts into one ZIP for easy download.
import zipfile, os
OUT_ZIP = '/content/fish_id_artifacts.zip'
with zipfile.ZipFile(OUT_ZIP, 'w') as z:
  for name in ['fish_id_model.tflite', 'fish_id_labels.json', 'fish_id_metrics.json']:
    p = f'/content/artifacts/{name}'
    assert os.path.exists(p), f'missing {p}'
    z.write(p, arcname=name)
print('Wrote', OUT_ZIP, os.path.getsize(OUT_ZIP)/1024, 'KB')

# Download to your machine.
from google.colab import files
files.download(OUT_ZIP)

## Next: upload to admin

Back in the admin: **Training → Models → Upload artifacts**. Unzip
the file this notebook downloaded and drag the three files onto
the upload area.

Look at the evaluation view for the new version:

- **Overall accuracy** — sanity check
- **Per-species accuracy** — anything below ~0.5 means that species
  needs more/better photos
- **Lookalike-group breakdown** — this is the pass/fail metric.
  A model that confuses Red vs. Vermilion Snapper is NOT shippable.

Only promote to production if the lookalike groups look clean.
